# Kali-Wireless TUI — Remote 15B Unrestricted Model (Colab T4 + ngrok)

**Model:** `yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF`

This notebook runs the strong **Unrestricted 15B** model on Colab T4 and exposes it via ngrok so your Kali TUI (on the droplet) can use it as a remote backend.

### Instructions
1. Runtime → Change runtime type → **T4 GPU** (High-RAM recommended)
2. Add your ngrok token as Colab Secret named `NGROK_AUTH_TOKEN`
3. Run cells in order
4. Copy the final ngrok URL and put it in your `docker-compose.remote-colab.yml`

In [ ]:
#@title 1. Install Ollama + dependencies
!apt-get update -qq && apt-get install -y -qq zstd 2>/dev/null || true
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q huggingface_hub pyngrok
print("[+] Ollama + dependencies installed")
!ollama --version

In [ ]:
#@title 2. Start Ollama server (with GPU optimizations)
import subprocess, os, time, requests

!pkill -f 'ollama serve' || true
time.sleep(2)

env = os.environ.copy()
env.update({
    "OLLAMA_HOST": "0.0.0.0:11434",
    "OLLAMA_FLASH_ATTENTION": "1",
    "CUDA_VISIBLE_DEVICES": "0"
} )

print("[*] Starting Ollama...")
subprocess.Popen(['ollama', 'serve'], env=env)

for i in range(25):
    try:
        if requests.get('http://localhost:11434/api/tags', timeout=2).status_code == 200:
            print("[+] Ollama is running on :11434")
            break
    except:
        pass
    time.sleep(1)
else:
    print("[!] Ollama failed to start")

In [ ]:
#@title 3. Download GGUF + Create Model with Exact Name
from huggingface_hub import hf_hub_download
import subprocess, os

repo_id = "yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF"
filename = "unrestricted-knowledge-will-not-refuse-15b-q4_k_m.gguf"
DESIRED_NAME = "yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF"

print(f"[*] Downloading {filename}... (this will take a while)")
model_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    cache_dir="/content/model_download"
)
print(f"[+] Downloaded to {model_path}")

print(f"[*] Creating Modelfile...")
with open("Modelfile", "w") as f:
    f.write(f"FROM {model_path}\n")

print(f"[*] Creating model as: {DESIRED_NAME}")
!ollama create {DESIRED_NAME} -f Modelfile

print("\n[+] Current models:")
!ollama list

print(f"\n[*] Warming up {DESIRED_NAME}...")
!ollama run {DESIRED_NAME} "System ready." --verbose 2>&1 | tail -5

In [ ]:
#@title 4. Start ngrok
from google.colab import userdata
import subprocess, time, requests

NGROK_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
!ngrok config add-authtoken {NGROK_TOKEN}
!pkill -f ngrok || true
time.sleep(1)

print("[*] Starting ngrok...")
subprocess.Popen(['ngrok', 'http', '11434'])
time.sleep(7)

r = requests.get('http://localhost:4040/api/tunnels', timeout=10)
url = [t['public_url'] for t in r.json()['tunnels'] if t['proto'] == 'https'][0]

print("\n" + "="*70)
print("✅ SUCCESS")
print("="*70)
print(f"\nPUBLIC OLLAMA URL:\n{url}")
print(f"\nUse these in docker-compose.remote-colab.yml:")
print(f"  OLLAMA_HOST={url}")
print(f"  MISTRAL_MODEL=yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF")
print("="*70)

with open('/content/remote_ollama_url.txt', 'w') as f:
    f.write(url)

In [ ]:
#@title 5. Keep-alive (run this in a separate cell and leave it running)
import time, requests
from datetime import datetime

print("🔄 Keep-alive running. This prevents Colab from disconnecting.")
print("Stop this cell when you are done.")

while True:
    try:
        requests.get('http://localhost:11434/api/tags', timeout=5)
        print(f"[{datetime.now().strftime('%H:%M')}] Still alive")
    except:
        pass
    time.sleep(300)

## Final Steps

1. Copy the **PUBLIC OLLAMA URL** printed above.
2. On your droplet:
   ```bash
   cd /root/kali-mistral-tui
   nano docker-compose.remote-colab.yml
   ```
3. Set:
   ```yaml
   - OLLAMA_HOST=https://your-ngrok-url.ngrok-free.dev
   - MISTRAL_MODEL=yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF
   ```
4. Restart the TUI:
   ```bash
   docker compose -f docker-compose.remote-colab.yml down
   docker compose -f docker-compose.remote-colab.yml up -d
   ```

**Note:** The model is heavy (15B). If you get OOM, you may need to fall back to a smaller model or use the 7B version.